# LifeMate — Fine-tune Qwen3-4B no thinking với LoRA trên A100 40 GB

1. Chọn **Runtime → Change runtime type → A100 GPU**.
2. Đặt `DATA_PATH` trỏ tới thư mục đã giải nén chứa `manifest.json`, `train.jsonl`
   và `validation.jsonl`. Mặc định đọc từ Drive. Chọn `local_folder` cho thư mục trong
   `/content`. Sau đó chạy tất cả cell.
3. Cuối notebook, Colab sẽ bắt đầu tải ZIP checkpoint qua trình duyệt.

Tập train có **5.794 dòng SFT** từ 2.136 episode. Tập dev BTC có **993 dòng SFT**
từ 309 episode. Notebook clone commit GitHub cố định và dùng trình kiểm tra hành
động của repo. Prompt đã được render theo Qwen3 no thinking. Loss tính trên
completion, gồm một tool call hoặc `DONE` và EOS.

Cấu hình A100 40 GB dùng LoRA BF16, microbatch 1, gradient checkpointing và
tích lũy gradient. Qwen chỉ tạo logits tại các vị trí dự đoán completion. Notebook
giữ đủ context, kiểm tra bộ nhớ trên mẫu train dài nhất rồi train 1 epoch và xuất adapter. Chọn
`EXPORT_MODE="merged"` để xuất model độc lập đã merge LoRA. Đặt
`SMOKE_TEST=True` để thử pipeline với 8 dòng train và 4 dòng dev trong 2 bước.
Checkpoint từ chế độ smoke được đánh dấu riêng.

Tài liệu [SFTTrainer](https://huggingface.co/docs/trl/v0.24.0/sft_trainer),
[Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) và
[Colab files.download](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py).


In [ ]:
#@title 1. Cấu hình
DATA_SOURCE = "drive_folder" #@param ["drive_folder", "local_folder", "upload_zip", "drive_zip"]
DATA_PATH = "/content/drive/MyDrive/LifeMate/sft_lifemate_2026-11-08" #@param {type:"string"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/LifeMate/lora_runs" #@param {type:"string"}
EXPORT_MODE = "adapter" #@param ["adapter", "merged"]
AUTO_DOWNLOAD = True #@param {type:"boolean"}
SMOKE_TEST = False #@param {type:"boolean"}
EPOCHS = 1.0 #@param {type:"number"}
LEARNING_RATE = 0.0001 #@param {type:"number"}
GRAD_ACCUMULATION = 16 #@param {type:"integer"}
SAVE_STEPS = 100 #@param {type:"integer"}
SEED = 42

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/Zalo-AIC-2026-Lifemate.git"
REPO_BRANCH = "feat/synthetic-dataset"
REPO_COMMIT = "09103e83636c03cb6344dbbb148de614473ad6ff"
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"
DATASET_SHA256 = "a335c30bf066650659f04874f570ad0f15b87026a1bae60b04f4ee9bc63e95e7"
MANIFEST_SHA256 = "14f2a8f689a6b10a606c700ef28adb3b7149780288bdb5544677e49fd427e147"

assert DATA_SOURCE in {"drive_folder", "local_folder", "upload_zip", "drive_zip"}
assert EXPORT_MODE in {"adapter", "merged"}
assert EPOCHS > 0 and LEARNING_RATE > 0 and GRAD_ACCUMULATION > 0 and SAVE_STEPS > 0


In [ ]:
#@title 2. Kiểm tra GPU và cài phiên bản huấn luyện
from importlib.metadata import version
import os
import subprocess
import sys
import torch

if torch.cuda.device_count() != 1:
    raise RuntimeError("Chọn một GPU A100 trong Runtime settings.")
gpu = torch.cuda.get_device_properties(0)
if "A100" not in gpu.name or not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"Runtime hiện dùng {gpu.name}. Chọn A100 rồi chạy lại.")
print({"gpu": gpu.name, "vram_GiB": round(gpu.total_memory / 2**30, 1),
       "torch": torch.__version__, "cuda": torch.version.cuda})

pins = {
    "transformers": "4.57.6",
    "trl": "0.24.0",
    "peft": "0.18.0",
    "datasets": "4.4.1",
    "accelerate": "1.11.0",
}
loaded = [name for name in pins if name in sys.modules and
          getattr(sys.modules[name], "__version__", None) != pins[name]]
if loaded:
    raise RuntimeError(f"Khởi động lại runtime trước khi chạy cell cài gói: {loaded}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *(f"{name}=={v}" for name, v in pins.items())], check=True)
versions = {name: version(name) for name in pins}
assert versions == pins, versions


In [ ]:
#@title 3. Clone repo và tạo thư mục cho run
from datetime import datetime, timezone
from pathlib import Path
import uuid

run_mode = "smoke" if SMOKE_TEST else "full"
run_id = f"{run_mode}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}"
work = Path("/content/lifemate_lora") / run_id
work.mkdir(parents=True, exist_ok=False)
repo = work / "repo"
subprocess.run(
    ["git", "clone", "--quiet", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(repo)],
    check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
)
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", REPO_COMMIT], check=True)
repo_commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert repo_commit == REPO_COMMIT
sys.path.insert(0, str(repo / "src"))
from lifemate.agent import parse_action

if DATA_SOURCE.startswith("drive_") or SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
run_dir = (Path(DRIVE_OUTPUT_ROOT) / run_id) if SAVE_TO_DRIVE else (work / "run")
run_dir.mkdir(parents=True, exist_ok=False)
print("Git commit:", repo_commit)
print("Run:", run_dir)


In [ ]:
#@title 4. Đọc thư mục SFT đã giải nén và kiểm tra checksum
from zipfile import ZipFile
from pathlib import PurePosixPath
import hashlib
import json
import shutil

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

required = ("manifest.json", "train.jsonl", "validation.jsonl")
dataset_input = {"source": DATA_SOURCE}
dataset_dir = work / "sft"
dataset_dir.mkdir(exist_ok=True)

if DATA_SOURCE.endswith("_folder"):
    source_dir = Path(DATA_PATH)
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Thư mục dữ liệu không tồn tại: {source_dir}")
    if not (source_dir / "manifest.json").is_file():
        children = [p.parent for p in source_dir.glob("*/manifest.json")]
        if len(children) != 1:
            raise ValueError("DATA_PATH phải trỏ tới thư mục chứa manifest.json.")
        source_dir = children[0]
    for name in required:
        shutil.copyfile(source_dir / name, dataset_dir / name)
else:
    if DATA_SOURCE == "upload_zip":
        from google.colab import files
        upload_dir = work / "upload"
        upload_dir.mkdir(exist_ok=True)
        uploaded = files.upload(target_dir=str(upload_dir))
        if len(uploaded) != 1:
            raise ValueError("Chọn một file LifeMate_SFT_LoRA_2026-11-08.zip.")
        dataset_zip = upload_dir / Path(next(iter(uploaded))).name
        del uploaded
    else:
        dataset_zip = Path(DATA_PATH)
    dataset_input["zip_sha256"] = sha256_file(dataset_zip)
    if dataset_input["zip_sha256"] != DATASET_SHA256:
        raise ValueError("Checksum ZIP không khớp bộ SFT LifeMate đã chuẩn bị.")
    with ZipFile(dataset_zip) as archive:
        members = archive.namelist()
        manifests = [n for n in members if PurePosixPath(n).name == "manifest.json"]
        if len(manifests) != 1:
            raise ValueError("ZIP cần có đúng một manifest.json.")
        base = PurePosixPath(manifests[0]).parent
        for name in required:
            member = str(base / name)
            if members.count(member) != 1:
                raise ValueError(f"Thiếu hoặc trùng file {member}")
            with archive.open(member) as source, (dataset_dir / name).open("wb") as target:
                shutil.copyfileobj(source, target)

dataset_input["manifest_sha256"] = sha256_file(dataset_dir / "manifest.json")
if dataset_input["manifest_sha256"] != MANIFEST_SHA256:
    raise ValueError("Manifest không khớp bộ SFT LifeMate đã chuẩn bị.")
spec = json.loads((dataset_dir / "manifest.json").read_text())
for name in ("train.jsonl", "validation.jsonl"):
    if sha256_file(dataset_dir / name) != spec["files"][name]:
        raise ValueError(f"Checksum không khớp: {name}")
assert spec["model"] == MODEL_ID and spec["tokenizer_revision"] == MODEL_REVISION
assert spec["enable_thinking"] is False and spec["completion_only_loss"] is True
assert spec["eos_token"] == "<|im_end|>" and spec["context_limit"] == 16384
print({part: spec["partitions"][part]["decisions"] for part in ("train", "validation")})


In [ ]:
#@title 5. Kiểm tra từng completion và tạo loss mask
from datasets import load_dataset
from transformers import AutoTokenizer, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
assert tokenizer.eos_token == spec["eos_token"]
assert tokenizer.pad_token_id is not None
tokenizer.padding_side = "right"
raw = load_dataset("json", data_files={
    part: str(dataset_dir / f"{part}.jsonl") for part in ("train", "validation")
}, cache_dir=str(work / "datasets_cache"))
for part in ("train", "validation"):
    assert len(raw[part]) == spec["partitions"][part]["decisions"], part
assert set(raw["train"]["episode_id"]).isdisjoint(raw["validation"]["episode_id"])
assert set(raw["train"]["split_group"]).isdisjoint(raw["validation"]["split_group"])

def longest_indices(dataset, count):
    lengths = [p + c for p, c in zip(dataset["prompt_tokens"], dataset["completion_tokens"])]
    return sorted(range(len(dataset)), key=lengths.__getitem__, reverse=True)[:count]

if SMOKE_TEST:
    raw["train"] = raw["train"].select(longest_indices(raw["train"], 8))
    raw["validation"] = raw["validation"].select(longest_indices(raw["validation"], 4))

def encode_batch(batch):
    prompts = batch["prompt"]
    completions = batch["completion"]
    p_ids = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        [p + c for p, c in zip(prompts, completions)], add_special_tokens=False
    )["input_ids"]
    masks = []
    for i, (prompt_ids, ids) in enumerate(zip(p_ids, full_ids)):
        prefix, target = prompts[i], completions[i]
        row_id = batch["id"][i]
        if not prefix.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"):
            raise ValueError(f"{row_id}: prefix không ở chế độ no thinking.")
        if not target.endswith(tokenizer.eos_token):
            raise ValueError(f"{row_id}: thiếu EOS.")
        parse_action(target[:-len(tokenizer.eos_token)])
        if ids[:len(prompt_ids)] != prompt_ids or ids[-1] != tokenizer.eos_token_id:
            raise ValueError(f"{row_id}: sai ranh giới token prompt/completion.")
        target_count = len(ids) - len(prompt_ids)
        if (len(prompt_ids) != batch["prompt_tokens"][i]
                or target_count != batch["completion_tokens"][i]
                or len(ids) > spec["context_limit"] or target_count < 2):
            raise ValueError(f"{row_id}: độ dài token không khớp manifest xuất dữ liệu.")
        masks.append([0] * len(prompt_ids) + [1] * target_count)
    return {"input_ids": full_ids, "completion_mask": masks}

tokenized = {
    part: ds.map(encode_batch, batched=True, batch_size=32, remove_columns=ds.column_names,
                 desc=f"Tokenize {part}")
    for part, ds in raw.items()
}
print({part: len(ds) for part, ds in tokenized.items()})


In [ ]:
#@title 6. Tạo LoRA BF16 và kiểm tra bộ nhớ với mẫu dài nhất
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.bfloat16,
    device_map={"": 0}, attn_implementation="sdpa",
)
base_model.config.use_cache = False
lora_config = LoraConfig(
    task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(base_model, lora_config)
pad_and_mask = DataCollatorForLanguageModeling(
    pad_token_id=tokenizer.pad_token_id, completion_only_loss=True,
)

def collate_completion(examples):
    if len(examples) != 1:
        raise ValueError("Cấu hình này dùng microbatch 1 cho cả train và eval.")
    batch = pad_and_mask(examples)
    # Vị trí t dự đoán token t+1. Giữ labels đầy đủ để Trainer đếm
    # đúng token khi tích lũy gradient, và gửi nhãn đã dịch cho loss Qwen.
    shifted = batch["labels"][0, 1:]
    positions = torch.where(shifted.ne(-100))[0]
    if positions.numel() == 0:
        raise ValueError("Completion không có token được giám sát.")
    batch["logits_to_keep"] = positions
    batch["shift_labels"] = shifted.index_select(0, positions).unsqueeze(0)
    return batch

training_args = TrainingArguments(
    output_dir=str(run_dir / "checkpoints"),
    seed=SEED, data_seed=SEED, bf16=True,
    num_train_epochs=EPOCHS, max_steps=2 if SMOKE_TEST else -1,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=1 if SMOKE_TEST else GRAD_ACCUMULATION,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch", learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
    warmup_ratio=0.03, max_grad_norm=1.0, remove_unused_columns=False,
    eval_strategy="no", prediction_loss_only=True,
    save_strategy="steps", save_steps=1 if SMOKE_TEST else SAVE_STEPS,
    save_total_limit=2, save_safetensors=True,
    logging_steps=1 if SMOKE_TEST else 10, report_to=[],
)
trainer = Trainer(
    model=model, args=training_args, data_collator=collate_completion,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
)
assert trainer.model_accepts_loss_kwargs
sample = tokenized["train"][longest_indices(raw["train"], 1)[0]]
batch = trainer.data_collator([sample])
expected = [token if active else -100 for token, active
            in zip(sample["input_ids"], sample["completion_mask"])]
assert batch["labels"][0].tolist() == expected
assert batch["shift_labels"][0].tolist() == [v for v in expected if v != -100]
assert batch["shift_labels"][0, -1] == tokenizer.eos_token_id
trainable, total = model.get_nb_trainable_parameters()
assert 0 < trainable < total < 5_000_000_000
print({"trainable_parameters": trainable, "total_parameters": total,
       "longest_train_tokens": len(sample["input_ids"]),
       "logit_positions_for_that_row": batch["logits_to_keep"].numel()})

# Thử forward/backward với mẫu dài nhất và cùng dtype/checkpointing như train.
# Lượt thử chỉ tính gradient. Reset gradient và seed trước khi bắt đầu train.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.train()
torch.cuda.reset_peak_memory_stats(0)
probe = {key: tensor.to(model.device) for key, tensor in batch.items()}
with trainer.accelerator.autocast():
    loss = model(**probe).loss
if not torch.isfinite(loss):
    raise RuntimeError("Loss thử không hữu hạn.")
loss.backward()
torch.cuda.synchronize(0)
preflight_peak_gib = round(torch.cuda.max_memory_reserved(0) / 2**30, 2)
model.zero_grad(set_to_none=True)
del loss, probe
set_seed(SEED)
print({"preflight_peak_reserved_GiB": preflight_peak_gib})


In [ ]:
#@title 7. Train LoRA và đo loss trên dev BTC
train_metrics = trainer.train().metrics
eval_metrics = trainer.evaluate()
trainer.save_metrics("train", train_metrics)
trainer.save_metrics("eval", eval_metrics)
print({"train_loss": train_metrics.get("train_loss"), "dev_btc_loss": eval_metrics["eval_loss"]})


In [ ]:
#@title 8. Lưu checkpoint và đóng gói ZIP
import gc
from zipfile import ZIP_STORED

adapter_dir = run_dir / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(adapter_dir)
if not (adapter_dir / "adapter_model.safetensors").is_file():
    raise RuntimeError("Không tìm thấy trọng số LoRA sau khi lưu.")

export_dir = adapter_dir
if EXPORT_MODE == "merged":
    gc.collect()
    torch.cuda.empty_cache()
    export_dir = run_dir / "merged_model"
    merged = trainer.model.merge_and_unload(safe_merge=True)
    merged.config.use_cache = True
    merged.save_pretrained(export_dir, safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(export_dir)
    if not list(export_dir.glob("model*.safetensors")):
        raise RuntimeError("Không tìm thấy trọng số model đã merge.")

metadata = {
    "mode": run_mode, "export_mode": EXPORT_MODE,
    "base_model": MODEL_ID, "model_revision": MODEL_REVISION,
    "github_commit": repo_commit, "dataset_input": dataset_input,
    "preflight_peak_reserved_GiB": preflight_peak_gib,
    "dataset_rows": {part: len(ds) for part, ds in tokenized.items()},
    "inference_chat_template_kwargs": {"enable_thinking": False},
    "versions": {**versions, "torch": torch.__version__},
    "gpu": gpu.name, "trainable_parameters": trainable, "total_parameters": total,
    "train_metrics": train_metrics, "dev_btc_metrics": eval_metrics,
}
(export_dir / "lifemate_run.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
)
archive_path = run_dir / f"lifemate_qwen3_4b_{run_mode}_{EXPORT_MODE}_{run_id}.zip"
temporary = archive_path.with_suffix(".zip.part")
try:
    with ZipFile(temporary, "x", ZIP_STORED) as archive:
        for path in sorted(export_dir.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(run_dir))
    with ZipFile(temporary) as archive:
        corrupt = archive.testzip()
        if corrupt:
            raise RuntimeError(f"ZIP bị lỗi ở {corrupt}")
    temporary.replace(archive_path)
finally:
    temporary.unlink(missing_ok=True)
print("File:", archive_path)
print("Dung lượng GiB:", round(archive_path.stat().st_size / 2**30, 3))
print("SHA256:", sha256_file(archive_path))


In [ ]:
#@title 9. Tải checkpoint về máy
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(archive_path))


Khi chạy inference, dùng tokenizer trong ZIP cùng `enable_thinking=False`.
Chế độ adapter nạp cùng Qwen3-4B tại revision ghi trong `lifemate_run.json`.
Chế độ merged nạp trực tiếp thư mục `merged_model` bằng Transformers.
`eval_loss` đo trên các quyết định của dev BTC. Điểm thực thi end-to-end cần
chạy agent trên ToolEnv BTC.

Có thể chạy lại cell cuối để kích hoạt lại lượt tải trên trình duyệt. Nếu bật
`SAVE_TO_DRIVE`, các checkpoint và ZIP nằm trong thư mục run trên Drive.
